In [11]:
import pandas
import glob
import pickle

In [35]:
data = pd.read_csv('../../data/holmes/fuse-org-negation/samples.csv')
data[data['set-0'] == 'test']

,inputs,context,topic,org_label,set-0,id,label
1546,('While Pcs have tons of hardware upgrades (vi...,NaN,NaN,1,test,43,1
1547,"('No.',)",NaN,NaN,1,test,106,1
1548,"(""There are two speakers up near the display a...",NaN,NaN,1,test,111,1
1549,"(""Again , no need to keep track of endless acc...",NaN,NaN,1,test,120,1
1550,('Battery life is not something I am concerned...,NaN,NaN,1,test,127,1
...,...,...,...,...,...,...,...
4043,"('This time it \x92s from The Clipse song , Le...",NaN,NaN,0,test,4474,0
4044,"(""For a large skillet meal , I 'd go with a li...",NaN,NaN,0,test,8529,0
4045,('I always use a spray-on shortening when I co...,NaN,NaN,0,test,9045,0
4046,"('To top it all off , listening to valid point...",NaN,NaN,0,test,3601,0


In [36]:
path = glob.glob('../../results/holmes/fuse-org-negation/facebook__bart-base/full/NONE/4048/**/0/done/preds.csv')
path2 = glob.glob('../../results/holmes/fuse-org-negation/microsoft__deberta-v3-base//full/NONE/4048/**/0/done/preds.csv')

In [37]:
files = pd.concat([pd.read_csv(file) for file in path])
files2 = pd.concat([pd.read_csv(file) for file in path2])

In [38]:
average_scores = files2.groupby('Unnamed: 0')['pred'].mean().reset_index()

The predictions from the dataframe under here should go into the app, since they are the averages for every sentence for fuse-org-negation, but for one ml model only

In [39]:
average_scores

,Unnamed: 0,pred
0,0,1.0
1,1,1.0
2,2,1.0
3,3,1.0
4,4,1.0
...,...,...
2497,2497,1.0
2498,2498,1.0
2499,2499,1.0
2500,2500,1.0


In [48]:
loaded_dump = pd.read_pickle('../../dumps/holmes/facebook__bart-base__full__NONE__probe-fuse-org-negation__4048__False.pickle')
loaded_dump[0]

{-1:                                                  inputs context  topic  \
 0     (The worst thing is it 's terribly slow boot s...            NaN   
 1     (It wo n't even let me shut it down , it will ...            NaN   
 2     (It also wo n't even allow me to burn a back u...            NaN   
 989           (Anyway do n't say I did n't warn you .,)            NaN   
 3     (Sure it looks nice but we are talking about a...            NaN   
 ...                                                 ...     ...    ...   
 1043                    (This is far from the truth !,)            NaN   
 1357  (I went shopping for 2 weeks of groceries for ...            NaN   
 1981             (Space is definitely not a problem .,)            NaN   
 1542  (With today 's gas prices going up this is a m...            NaN   
 3184  (My first fill up , after driving about 110 hi...            NaN   
 
       org_label  set-0     id  label  \
 0             1  train      3      1   
 1          

In [50]:
def load_probe_file(probe_file:str, sample_size=0):
    loaded_frame = pd.read_csv(probe_file).sort_values("id")

    if sample_size > 0 and sample_size < loaded_frame.shape[0]:
        loaded_frame = pd.concat([
            loaded_frame[loaded_frame["set-0"] == "train"].sample(sample_size),
            loaded_frame[loaded_frame["set-0"] == "dev"],
            loaded_frame[loaded_frame["set-0"] == "test"],
        ])

    if not "id" in loaded_frame.columns:
        loaded_frame["id"] = loaded_frame.index

    if str(loaded_frame["context"].values[0]) == "nan":
        loaded_frame.loc[:,"context"] = ""
    loaded_frame.loc[:,"inputs"] = loaded_frame["inputs"].apply(lambda ele: eval(ele))
    return loaded_frame

In [4]:
import torch
from sentence_transformers import SentenceTransformer
from sentence_transformers.models import Transformer

from defs.control_task_types import CONTROL_TASK_TYPES
from model.EightBitTransformer import EightBitTransformer
from model.FourBitTransformer import FourBitTransformer
from model.HalfPrecisionTransformer import HalfPrecisionTransformer
from model.ParallelSentenceTransformer import ParallelSentenceTransformer
from model.transformer_adaptions import BartTransformer
from utils.SpecificLayerPooling import SpecificLayerPooling
from utils.experiment_util import init_random_weights

def load_model(model_name, control_task_type, encoding, scalar_mixin=False):

    if "glove" in model_name:
        base_model = SentenceTransformer(model_name)
    else:

        device = "cuda" if torch.cuda.is_available() else "cpu"

        if "bart" in model_name:
            transformer = BartTransformer(model_name, model_args={"output_hidden_states": True})
        elif encoding == "half":
            transformer = HalfPrecisionTransformer(model_name, model_args={"output_hidden_states": True})
        elif encoding == "four_bit":
            transformer = FourBitTransformer(model_name, model_args={"output_hidden_states": True})
        elif encoding == "eight_bit":
            transformer = EightBitTransformer(model_name, model_args={"output_hidden_states": True})
        else:
            transformer = Transformer(model_name, model_args={"output_hidden_states": True})

        if control_task_type == CONTROL_TASK_TYPES.RANDOM_WEIGHTS and model_name in ["t5-base", "roberta-base", "microsoft/deberta-base", "microsoft/deberta-v3-base", "bert-base-uncased", "albert-base-v2", "google/electra-base-discriminator"]:
            transformer.auto_model.encoder.apply(init_random_weights)
        elif control_task_type == CONTROL_TASK_TYPES.RANDOM_WEIGHTS and model_name in ["facebook/bart-base"]:
            transformer.auto_model.encoder.apply(init_random_weights)
            transformer.auto_model.decoder.apply(init_random_weights)
        elif control_task_type == CONTROL_TASK_TYPES.RANDOM_WEIGHTS and model_name in ["gpt2"]:
            transformer.auto_model.h.apply(init_random_weights)

        word_embedding_dimension = transformer.get_word_embedding_dimension()

        pooling = SpecificLayerPooling(
            word_embedding_dimension=word_embedding_dimension,
            layers=[-1], pooling_mode="mean"
        )

        base_model = ParallelSentenceTransformer(modules=[transformer,pooling])

        if device == "cuda" and "cuda" not in str(base_model.device):
            base_model = base_model.to(device)


        if base_model.tokenizer.pad_token is None:
            base_model.tokenizer.pad_token = base_model.tokenizer.eos_token

    return base_model

In [13]:
from defs.probe_task_types import PROBE_TASK_TYPES
import os

def load_folds(probe_frame:pandas.DataFrame, base_model:SentenceTransformer, probe_task_type:PROBE_TASK_TYPES, encoding_batch_size=10, encoding="full"):
    encoded_folds = []
    os.environ["TOKENIZERS_PARALLELISM"] = "false"
    encoded_folds.append(
        encode_fold_inputs(
            probe_frame=probe_frame,
            base_model=base_model,
            probe_task_type=probe_task_type,
            encoding=encoding,
            encoding_batch_size=encoding_batch_size
        )
    )
    return encoded_folds


In [14]:
def encode_fold_inputs(probe_frame:pandas.DataFrame, probe_task_type:PROBE_TASK_TYPES, base_model:SentenceTransformer, encoding_batch_size=10, encoding="full"):

    inputs_encoded = []

    for frame in tqdm(numpy.array_split(probe_frame, 5)):

        chunk = encode_inputs(
            inputs=list(frame["inputs"]),
            context=list(frame["context"]),
            base_model=base_model,
            probe_task_type=probe_task_type,
            encoding=encoding,
            encoding_batch_size=encoding_batch_size
        )

        inputs_encoded.append(chunk)
        gc.collect()

    inputs_encoded = {
        layer: list(itertools.chain.from_iterable([chunk[layer] for chunk in inputs_encoded]))
        for layer in inputs_encoded[0].keys()
    }

    fold_frames = {}
    probe_frame["context"] = ""
    for layer, layer_inputs_encoded in inputs_encoded.items():
        layer_fold_frame = probe_frame.copy()
        layer_fold_frame["inputs_encoded"] = layer_inputs_encoded
        fold_frames[layer] = layer_fold_frame

    return fold_frames

In [ ]:
def encode_inputs(inputs:List[List[str]], context:List[str], base_model:SentenceTransformer, probe_task_type:PROBE_TASK_TYPES, encoding_batch_size=10, encoding="full"):
    try:
        model_type = type(base_model[0].auto_model).__name__
        output_value = "token_layer_embeddings"
    except:
        model_type = "glove"
        output_value = "token_embeddings"

    if not hasattr(base_model.tokenizer, "sep_token") or base_model.tokenizer.sep_token is None:
        sep_token = ""
    else:
        sep_token = base_model.tokenizer.sep_token

    device = "cuda" if torch.cuda.is_available() else "cpu"

    if probe_task_type in [PROBE_TASK_TYPES.SENTENCE_TOKENS, PROBE_TASK_TYPES.SENTENCE_PAIR_TOKENS_BI, PROBE_TASK_TYPES.SENTENCE_PAIR_TOKENS_CROSS]:

        if probe_task_type in [PROBE_TASK_TYPES.SENTENCE_TOKENS, PROBE_TASK_TYPES.SENTENCE_PAIR_TOKENS_BI]:
            reduced_context = list(set(itertools.chain(*[eval(ele) for ele in context])))
        else:
            reduced_context = list(set([(" " + sep_token + " ").join(eval(ele)) for ele in context]))


        encoded_context = base_model.encode(
            sentences=reduced_context, show_progress_bar=True, batch_size=encoding_batch_size,
            output_value=output_value, convert_to_numpy=True, device=device
        )

        if model_type == "glove":
            encoded_context = {0: encoded_context}

        encoded_context_dict = {
            layer:dict(zip(reduced_context, layer_encoded_inputs))
            for layer, layer_encoded_inputs in encoded_context.items()
        }

        del encoded_context

        context = [
            eval(entry)
            for entry in context
        ]

        if model_type == "glove":
            pre_tokenized_context = [
                [base_model.tokenizer.tokenize(ele) for ele in entry]
                for entry in tqdm(context)
            ]
        else:
            pre_tokenized_context = [
                [base_model.tokenizer(ele) for ele in entry]
                for entry in tqdm(context)
            ]


        encoded_inputs = [
                [
                    extract_embeddings_for_input_element(
                        input_element, context_ele, pre_tokenized_context_ele, encoded_context_dict, base_model, probe_task_type, model_type, encoding
                    )
                    for input_element in input_list
                ]
            for input_list, context_ele, pre_tokenized_context_ele in tqdm(zip(inputs, context, pre_tokenized_context))
        ]

        del context
        del pre_tokenized_context

        return {
            layer: [
                [input_element[i] for input_element in encoded_input]
                for encoded_input in encoded_inputs
            ] for i, layer in enumerate(encoded_context_dict.keys())
        }
    elif probe_task_type in [PROBE_TASK_TYPES.SENTENCE_SPANS]:

        if probe_task_type in [PROBE_TASK_TYPES.SENTENCE_SPANS]:
            reduced_context = list(set(itertools.chain(*[eval(ele) for ele in context])))
        else:
            reduced_context = list(set([(" " + sep_token + " ").join(eval(ele)) for ele in context]))

        encoded_context = base_model.encode(
            sentences=reduced_context, show_progress_bar=True, batch_size=encoding_batch_size,
            output_value=output_value, convert_to_numpy=True, device=device
        )

        if model_type == "glove":
            encoded_context = {0: encoded_context}

        encoded_context_dict = {
            layer:dict(zip(reduced_context, layer_encoded_inputs))
            for layer, layer_encoded_inputs in encoded_context.items()
        }

        del encoded_context

        context = [
            eval(entry)
            for entry in context
        ]

        if model_type == "glove":
            pre_tokenized_context = [
                [base_model.tokenizer.tokenize(ele) for ele in entry]
                for entry in tqdm(context)
            ]
        else:
            pre_tokenized_context = [
                [base_model.tokenizer(ele) for ele in entry]
                for entry in tqdm(context)
            ]

        n_inputs = len(set([ele[-1] for ele in inputs[0]]))

        dimension_encodings = []

        for input_index in range(n_inputs):
            filtered_input = [
                tuple([input_element[:-1] for input_element in input_list if input_element[-1] == input_index])
                for input_list in inputs
            ]

            encoded_filtered_inputs = [
                [
                    extract_embeddings_for_input_element(
                        input_element, context_ele, pre_tokenized_context_ele, encoded_context_dict, base_model, probe_task_type, model_type, encoding
                    )
                    for input_element in input_list
                ]
                for input_list, context_ele, pre_tokenized_context_ele in tqdm(zip(filtered_input, context, pre_tokenized_context))
            ]

            dimension_encodings.append([numpy.array(ele).mean(axis=0) for ele in encoded_filtered_inputs])

        del context
        del pre_tokenized_context

        return {
            layer: [
                [dimension_encodings[input_index][j][i] for input_index in range(n_inputs)]
                for j in range(len(encoded_filtered_inputs))
            ] for i, layer in enumerate(encoded_context_dict.keys())
        }

    else:

        if probe_task_type in [PROBE_TASK_TYPES.SENTENCE, PROBE_TASK_TYPES.SENTENCE_PAIR_BI]:

            flatten_inputs = list(set(itertools.chain(*inputs)))

            encoded_inputs = base_model.encode(
                sentences=flatten_inputs, show_progress_bar=True, batch_size=encoding_batch_size, device=device
            )

            if model_type == "glove":
                encoded_inputs = {0: encoded_inputs}

            encoded_inputs_dict = {
                layer:dict(zip(flatten_inputs, layer_encoded_inputs))
                for layer, layer_encoded_inputs in encoded_inputs.items()
            }

            if encoding == "half" or encoding == "four_bit":
                dtype = numpy.float
            else:
                dtype = numpy.float16

            return {
                layer: [numpy.array([layer_encoded_inputs_dict[ele] for ele in input]).astype(dtype) for input in inputs]
                for layer, layer_encoded_inputs_dict in encoded_inputs_dict.items()
            }

        elif probe_task_type in [PROBE_TASK_TYPES.SENTENCE_PAIR_CROSS]:

            flatten_inputs = [(" " + sep_token + " ").join(ele) for ele in inputs]
            encoded_inputs = base_model.encode(
                sentences=flatten_inputs, show_progress_bar=True, batch_size=encoding_batch_size, device=device
            )

            if model_type == "glove":
                encoded_inputs = {0: encoded_inputs}

            if encoding == "half" or encoding == "four_bit":
                dtype = numpy.float
            else:
                dtype = numpy.float16

            return {
                layer: [numpy.array(ele).astype(dtype) for ele in encoded_layer_inputs.tolist()]
                for layer, encoded_layer_inputs in encoded_inputs.items()
            }

In [ ]:
def extract_embeddings_for_input_element(input_element:str, context:str, pre_tokenized_context, layer_embeddings, base_model:SentenceTransformer, probe_task_type:PROBE_TASK_TYPES, model_type:str, encoding:str):
    input_token, input_element_index, input_token_start, input_token_end = input_element

    if not hasattr(base_model.tokenizer, "sep_token") or base_model.tokenizer.sep_token is None:
        sep_token = ""
    else:
        sep_token = base_model.tokenizer.sep_token


    device = "cuda" if torch.cuda.is_available() else "cpu"

    if probe_task_type in [PROBE_TASK_TYPES.SENTENCE_TOKENS, PROBE_TASK_TYPES.SENTENCE_SPANS, PROBE_TASK_TYPES.SENTENCE_PAIR_TOKENS_BI]:
        if model_type == "glove":
            input_token_tokenized = base_model.tokenizer.tokenize(input_token)
            if len(input_token_tokenized) == 0:
                return base_model.encode(["unk"], device=device)
            input_indices = find_sub_list_start(input_token_tokenized, pre_tokenized_context[input_element_index])
            if len(input_indices) > 0:
                input_indices = range(input_indices[0], input_indices[0] + len(input_token_tokenized))
            else:
                return base_model.encode(["unk"], device=device)
        else:
            try:
                input_indices = find_sub_list(input_token_start, input_token_end, pre_tokenized_context[input_element_index])
            except:
                print(pre_tokenized_context[input_element_index])

        embeddings = torch.stack([context_embeddings[context[input_element_index]] for layer, context_embeddings in layer_embeddings.items()])

    else:
        first_context, second_context = context

        joined_context = (" " + sep_token + " ").join(context)

        if input_element_index == 0:
            input_indices = find_sub_list(input_token_start, input_token_end, pre_tokenized_context)
        else:
            input_indices = find_sub_list(input_token_start + len(first_context) + 7, input_token_end + len(first_context) + 7, pre_tokenized_context)

        if not input_indices and model_type == "glove":
            return base_model.encode(["unk"], device=device)

        embeddings = torch.stack([context_embeddings[joined_context] for layer, context_embeddings in layer_embeddings.items()])

    try:
        selected_embeddings = embeddings[:, input_indices].mean(dim=1).cpu().detach().numpy()
    except:
        print()

    if encoding == "half" or encoding == "four_bit":
        selected_embeddings = selected_embeddings.astype(numpy.float16)

    return selected_embeddings


In [1]:
import sys
sys.path.append('../../src/')

In [2]:
from utils import data_loading

In [3]:
from defs.control_task_types import CONTROL_TASK_TYPES
from defs.probe_task_types import PROBE_TASK_TYPES

In [4]:
base_model = data_loading.load_model("facebook/bart-base", CONTROL_TASK_TYPES.NONE, "full")

/opt/miniconda3/envs/HolmesEvaluation/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [5]:
probe_frame = data_loading.load_probe_file('../../data/holmes/fuse-org-negation/samples.csv', CONTROL_TASK_TYPES.NONE)

In [6]:
probing_frames = data_loading.load_folds(
        probe_frame=probe_frame,
        base_model=base_model,
        probe_task_type= PROBE_TASK_TYPES.SENTENCE,
        encoding="full",
        encoding_batch_size=10,
    )


  0%|          | 0/5 [00:00<?, ?it/s]

Batches:   0%|          | 0/81 [00:00<?, ?it/s]

 20%|██        | 1/5 [00:09<00:37,  9.47s/it]

Batches:   0%|          | 0/81 [00:00<?, ?it/s]

 40%|████      | 2/5 [00:18<00:27,  9.13s/it]

...

 60%|██████    | 3/5 [00:27<00:18,  9.10s/it]

Batches:   0%|          | 0/81 [00:00<?, ?it/s]

 80%|████████  | 4/5 [00:36<00:09,  9.25s/it]

Batches:   0%|          | 0/81 [00:00<?, ?it/s]

100%|██████████| 5/5 [00:46<00:00,  9.24s/it]


In [7]:
probing_frames

[{-1:                                                  inputs context  topic  \
  0     (The worst thing is it 's terribly slow boot s...            NaN   
  1     (It wo n't even let me shut it down , it will ...            NaN   
  2     (It also wo n't even allow me to burn a back u...            NaN   
  989           (Anyway do n't say I did n't warn you .,)            NaN   
  3     (Sure it looks nice but we are talking about a...            NaN   
  ...                                                 ...     ...    ...   
  1043                    (This is far from the truth !,)            NaN   
  1357  (I went shopping for 2 weeks of groceries for ...            NaN   
  1981             (Space is definitely not a problem .,)            NaN   
  1542  (With today 's gas prices going up this is a m...            NaN   
  3184  (My first fill up , after driving about 110 hi...            NaN   
  
        org_label  set-0     id  label  \
  0             1  train      3      1 

In [8]:
loaded_probing_frames = data_loading.load_probing_frames(probing_frames, "full")

4048it [00:00, 54623.77it/s]


In [9]:
loaded_probing_frames

[{'train':                                                 inputs context  topic  \
  0    (The worst thing is it 's terribly slow boot s...            NaN   
  1    (It wo n't even let me shut it down , it will ...            NaN   
  2    (It also wo n't even allow me to burn a back u...            NaN   
  3    (Sure it looks nice but we are talking about a...            NaN   
  4    (For those who are looking to buy a computer t...            NaN   
  ..                                                 ...     ...    ...   
  266  (The main problem that faces almost every pair...            NaN   
  267  (Some songs dont even sound like the two part...            NaN   
  268  (The rock parts are also sloopily throw togeth...            NaN   
  269  (I have no clue why Everlast needed to re-make...            NaN   
  270  (Im not sure who Butch Vig is , but their ble...            NaN   
  
       org_label  set-0    id  label  \
  0            1  train     3      1   
  1     

@click.option('--config_file_path', type=str, default='../data/flash-holmes/protoroles-change_of_state/config-none.yaml')
@click.option('--model_name', type=str, default="bbunzeck/baby_llama")
@click.option('--model_precision', type=str, default="full")
@click.option('--seeds', type=str, default="0,1,2,3,4")
@click.option('--num_hidden_layers', type=str, default="0")
@click.option('--batch_size', type=int, default=16)
@click.option('--run_probe', type=bool, default=True)
@click.option('--run_mdl_probe', type=bool, default=True)
@click.option('--project_prefix', type=str, default="dev")
@click.option('--dump_preds', is_flag=True, default=True)
@click.option('--force', is_flag=True, default=False)
@click.option('--dump_folder', type=str, default="../dumps")
@click.option('--result_folder', type=str, default="../results")
@click.option('--logging', type=str, default="wandb")

In [37]:
from utils import config_loading
import yaml

file_stream = open('../../data/holmes/fuse-org-negation/config-none.yaml', "r")
config = yaml.safe_load(file_stream)

control_task_type = CONTROL_TASK_TYPES[config["control_task_type"]]

base_config = config_loading.load_base_config(
        config=config, encoding="full",
        seeds="0,1,2,3,4", num_hidden_layers="0",
        model_name="facebook/bart-base", batch_size=16,
        project_prefix="dev", control_task_type=control_task_type
    )

configs = config_loading.get_load_configs(
        base_config, True, True
    )

In [39]:
from probe import get_hyperparameters

hyperparameters = get_hyperparameters(config['hyperparameters'])
hyperparameters

[{'learning_rate': 0.001,
  'batch_size': 16,
  'optimizer': torch.optim.adam.Adam,
  'hidden_dim': 0,
  'dropout': 0.2,
  'warmup_rate': 0.1,
  'num_hidden_layers': 0,
  'seed': 0},
 {'learning_rate': 0.001,
  'batch_size': 16,
  'optimizer': torch.optim.adam.Adam,
  'hidden_dim': 0,
  'dropout': 0.2,
  'warmup_rate': 0.1,
  'num_hidden_layers': 0,
  'seed': 1},
 {'learning_rate': 0.001,
  'batch_size': 16,
  'optimizer': torch.optim.adam.Adam,
  'hidden_dim': 0,
  'dropout': 0.2,
  'warmup_rate': 0.1,
  'num_hidden_layers': 0,
  'seed': 2},
 {'learning_rate': 0.001,
  'batch_size': 16,
  'optimizer': torch.optim.adam.Adam,
  'hidden_dim': 0,
  'dropout': 0.2,
  'warmup_rate': 0.1,
  'num_hidden_layers': 0,
  'seed': 3},
 {'learning_rate': 0.001,
  'batch_size': 16,
  'optimizer': torch.optim.adam.Adam,
  'hidden_dim': 0,
  'dropout': 0.2,
  'warmup_rate': 0.1,
  'num_hidden_layers': 0,
  'seed': 4}]

In [44]:
input_dim = loaded_probing_frames[0]["train"].iloc[0]["inputs_encoded"].shape[-1]


In [47]:
from model.probing_model import LinearProbingModel
from torch.optim import Adam


probing_model = LinearProbingModel

In [12]:
from defs import default_config

In [23]:
default_config.DEFAULT_CONFIG['hyperparameters']

{'learning_rate': [0.001],
 'batch_size': [64],
 'optimizer': [torch.optim.adam.Adam],
 'hidden_dim': [0, 1000],
 'dropout': [0.2],
 'warmup_rate': [0.1],
 'num_hidden_layers': [0, 1, 2]}

In [49]:
probing_model.load_from_checkpoint('../../results/holmes/fuse-org-negation/facebook__bart-base/full/NONE/4048/0/0/done/epoch=0-step=17.ckpt', {'learning_rate': 0.001,
                                                   'num_labels': 2,
                                                   'input_dim': input_dim,                           
  'batch_size': 16,
  'optimizer': Adam,
  'hidden_dim': 0,
  'dropout': 0.2,
  'warmup_rate': 0.1,
  'num_hidden_layers': 0,
  'seed': 2})

RuntimeError: Attempting to deserialize object on a CUDA device but torch.cuda.is_available() is False. If you are running on a CPU-only machine, please use torch.load with map_location=torch.device('cpu') to map your storages to the CPU.